[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bernardorivas/wael/blob/main/notebooks/example2_mixed_2network.ipynb)

# Example 2. Mixed-feedback circuit

Given a parameter sample in DSGRN node 712, this notebook simulates the mixed-feedback circuit as a Hill system and, after adding noise, infers the parameters back with least squares and a PINN. Recovery is evaluated by DSGRN region membership.

In [ ]:
%pip install -q DSGRN
%pip install -q tqdm git+https://github.com/marciogameiro/DSGRN_utils.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy.stats import qmc
import torch, torch.nn as nn
import DSGRN, DSGRN_utils
np.set_printoptions(precision=3, suppress=True)

# Train on a GPU when one is available: CUDA on Colab, MPS on Apple silicon, otherwise CPU.
# Note that MPS is single precision only, so CUDA is set to float32 for reproducibility.
if torch.cuda.is_available():
    DEVICE, TORCH_DTYPE = torch.device('cuda'), torch.float32
elif torch.backends.mps.is_available():
    DEVICE, TORCH_DTYPE = torch.device('mps'), torch.float32
else:
    DEVICE, TORCH_DTYPE = torch.device('cpu'), torch.float64
torch.set_default_dtype(TORCH_DTYPE)
print('torch device:', DEVICE, '| dtype:', TORCH_DTYPE)

### Hill functions

The production is built from Hill functions
\begin{equation}
f^+(x,L,U,\theta,d) = L + (U-L) \frac{x^d}{x^d+\theta^d}, \quad
f^-(x,L,U,\theta,d) = L + (U-L) \frac{\theta^d}{x^d+\theta^d}
\end{equation}
where
*   `L` and `U` are lower and upper values,
*   `theta` is a threshold value,
*   `d` is the steepness coefficient.

We use `f+` when it is activating and `f-` when it is repressing, noting that $f^++f^-=L+U$.

We define `hill_act` and `hill_rep` below, both in NumPy for simulation and in torch for the PINN's physics loss.

In [ ]:
# --- Hill production functions: smooth switches (gamma normalized to 1) ---
def hill_act(x, L, U, th, d):   # activating: low L -> high U as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * xd / (td + xd)

def hill_rep(x, L, U, th, d):   # repressing: high U -> low L as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * td / (td + xd)

In [ ]:
# --- the same two functions in torch (used inside the network's physics loss) ---
def hill_act_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * xd / (td + xd)

def hill_rep_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * td / (td + xd)

## 1. Model and DSGRN region (node 712)

Consider a two-gene circuit ($\gamma=1$): gene `x1` self-activates and is activated by `x2` (additive), while gene `x2` is repressed by `x1` and self-activates (multiplicative),

$$\dot{x}_1=-x_1+\underbrace{f^+(x_1)+f^+(x_2)}_{\text{additive}},\qquad
\dot{x}_2=-x_2+\underbrace{f^-(x_1)\,f^+(x_2)}_{\text{multiplicative}}.$$

The node-712 region is a chain of inequalities ordering the thresholds against sums of production levels (additive gene `x1`) and products of production levels (multiplicative gene `x2`). Membership is tested with DSGRN: `to_matrices` packs `(L, U, theta)` into DSGRN's parameter matrices, `par_index_from_sample` returns the region index, and `in_region` checks it against node 712.

In [ ]:
P = dict(L11=1.0,U11=4.0,th11=4.0,d11=10.0, L21=1.0,U21=2.0,th21=1.5,d21=10.0,
         L12=1.0,U12=2.0,th12=2.5,d12=10.0, L22=1.0,U22=4.0,th22=3.0,d22=10.0, g=1.0)

def rhs(t, x, p):
    x1, x2 = x
    prod1 = (hill_act(x1, p['L11'],p['U11'],p['th11'],p['d11'])
             + hill_act(x2, p['L21'],p['U21'],p['th21'],p['d21']))
    prod2 = (hill_rep(x1, p['L12'],p['U12'],p['th12'],p['d12'])
             * hill_act(x2, p['L22'],p['U22'],p['th22'],p['d22']))
    return [-p['g']*x1 + prod1, -p['g']*x2 + prod2]

NET_SPEC = 'x : x+y\ny : (~x)y\n'
_net = DSGRN.Network(NET_SPEC); _pg = DSGRN.ParameterGraph(_net)
TARGET = 712
def to_matrices(p):              # (L, U, theta) -> DSGRN L, U, T matrices, all indexed [source, target]
    # DSGRN entry T[i,j] = gamma_i * theta_{ij} (threshold on the source x_i; gamma = 1, so T = theta).
    # Node j is tested by ordering its production {L[i,j], U[i,j]} (inputs i->j) against its
    # OUTPUT thresholds T[j,k] (edges j->k):  L_{ij} < gamma_j theta_{jk} < U_{ij}.
    n = 2
    L = np.zeros((n, n)); U = np.zeros((n, n)); T = np.zeros((n, n))
    L[0,0],U[0,0]=p['L11'],p['U11']; L[1,0],U[1,0]=p['L21'],p['U21']
    L[0,1],U[0,1]=p['L12'],p['U12']; L[1,1],U[1,1]=p['L22'],p['U22']
    T[0,0],T[0,1],T[1,0],T[1,1]=p['th11'],p['th12'],p['th21'],p['th22']
    return L, U, T
def region_of(p):                # parameters -> DSGRN region index (-1 if outside)
    return DSGRN.par_index_from_sample(_pg, *to_matrices(p))
def in_region(p):                # membership in the target region
    return region_of(p) == TARGET

print('do the true parameters lie in node 712?', in_region(P), '| region', region_of(P))

In [ ]:
def simulate(rhs, p, x0, T, n):
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(p,), rtol=1e-9, atol=1e-11)
    return t, sol.y.T   # times (n,), states (n, m)

def add_noise(y, ub_frac, scale, rng):
    yn = y + rng.uniform(-ub_frac*scale, ub_frac*scale, size=y.shape)
    return np.clip(yn, 0.0, None)   # concentrations stay non-negative

## 2. Synthetic data

Ground truth parameters were chosen in node 712. The system is numerically integrated from several initial conditions and corrupted with bounded noise.

In [ ]:
T, n = 5.0, 80
MARGIN = 1.2
box_hi = [MARGIN*(P['U11'] + P['U21']), MARGIN*(P['U12'] * P['U22'])]
n_ic = 15
x0s = qmc.scale(qmc.LatinHypercube(d=2, seed=0, optimization='random-cd').random(n_ic), [0.0, 0.0], box_hi).tolist()
ts, xs = [], []
for x0 in x0s:
    t, y = simulate(rhs, P, x0, T, n); ts.append(t); xs.append(y)

gap = min(abs(P['L11']-P['U11']), abs(P['L21']-P['U21']),
          abs(P['L12']-P['U12']), abs(P['L22']-P['U22']))
rng = np.random.default_rng(0)
xs_noisy = [add_noise(y, 0.25, gap, rng) for y in xs]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for t, y, yn in zip(ts, xs, xs_noisy):
    ax[0].plot(t, yn[:,0], '.', ms=3); ax[0].plot(t, y[:,0], 'k-', lw=.6, alpha=.4)
    ax[1].plot(t, yn[:,1], '.', ms=3); ax[1].plot(t, y[:,1], 'k-', lw=.6, alpha=.4)
ax[0].set_title('$x_1$ (noisy)'); ax[1].set_title('$x_2$ (noisy)')
plt.tight_layout(); plt.show()

## 3. Least squares across noise

Least squares fits the twelve `(L, U, theta)` and four per-edge `d` from finite-difference slopes, with `gamma` fixed. Only the twelve `(L, U, theta)` are scored. The cell reports the region-recovery rate per noise level over repeated noisy datasets.

In [ ]:
KEYS = ['L11','U11','L21','U21','th11','th12','L12','U12','L22','U22','th21','th22']
DKEYS = ['d11','d21','d12','d22']
def ls_recover(ts, xs, g=1.0):
    X = np.vstack(xs)
    DX = np.vstack([np.gradient(y, t, axis=0) for t, y in zip(ts, xs)])
    scale = np.maximum(np.std(DX, axis=0), 1e-6)
    def resid(z):
        p = dict(zip(KEYS, z[:12])); p.update(dict(zip(DKEYS, z[12:])))
        f1 = (-g*X[:,0] + hill_act(X[:,0],p['L11'],p['U11'],p['th11'],p['d11'])
              + hill_act(X[:,1],p['L21'],p['U21'],p['th21'],p['d21']))
        f2 = (-g*X[:,1] + hill_rep(X[:,0],p['L12'],p['U12'],p['th12'],p['d12'])
              * hill_act(X[:,1],p['L22'],p['U22'],p['th22'],p['d22']))
        return np.concatenate([(DX[:,0]-f1)/scale[0], (DX[:,1]-f2)/scale[1]])
    z0 = np.array([1.5,3.,1.5,1.8,3.5,2.2,1.2,1.8,1.2,3.5,1.3,2.5] + [8.0]*4)
    s = least_squares(resid, z0, bounds=([0]*12+[0.25]*4, [30]*12+[80]*4),
                      max_nfev=20000, loss='soft_l1')
    p = dict(zip(KEYS, s.x[:12])); p.update(dict(zip(DKEYS, s.x[12:]))); p['g'] = g; return p

print('LS at 25% noise lands in node 712:', in_region(ls_recover(ts, xs_noisy)))

levels = [0.0, 0.1, 0.25, 0.5]; n_trials = 20; rate = []
for ub in levels:
    r = np.random.default_rng(7); hits = 0
    for _ in range(n_trials):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        hits += in_region(ls_recover(ts, xs_n))
    rate.append(hits / n_trials)
plt.plot([100*l for l in levels], rate, 'o-')
plt.xlabel('noise upper bound (%)'); plt.ylabel('region-recovery rate (LS)')
plt.ylim(-0.05, 1.05); plt.title('Mixed feedback: least squares drifts out of the region')
plt.show()
print('LS region-recovery rate by noise:', dict(zip(levels, rate)))

In [ ]:
def build_tensors(ts, xs, x0s):
    # flatten all trajectories into (N,1) time, (N,m) repeated x0, (N,m) measured states
    t_d = np.concatenate([t[:, None] for t in ts])
    x_d = np.vstack(xs)
    x0_d = np.vstack([np.tile(x0, (len(t), 1)) for x0, t in zip(x0s, ts)])
    t_ic = np.zeros((len(x0s), 1)); x_ic = np.array(x0s, dtype=float)  # t=0 anchors
    to = lambda a: torch.tensor(a, dtype=torch.get_default_dtype(), device=DEVICE)
    return (to(t_d), to(x0_d), to(x_d), to(t_ic), to(x_ic))

## 4. PINN

A single PINN run on this circuit jointly learns the same sixteen parameters as the least-squares baseline, from a neutral init, with `gamma` fixed at 1 in both methods. Only the twelve `(L, U, theta)` are scored. A full rate-vs-noise PINN sweep is omitted for cost; `steps` and a noise loop can be increased on a GPU.

In [ ]:
# --- the physics-informed network: surrogate u_theta(t, x0) + learnable (L,U,theta,d) ---
class PINN(nn.Module):
    def __init__(self, m, param_init, T, hidden=64, depth=4):
        super().__init__()
        self.m, self.T = m, T
        in_dim = 1 + m
        layers, d0 = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d0, hidden), nn.Tanh()]; d0 = hidden
        layers += [nn.Linear(d0, m)]
        self.net = nn.Sequential(*layers)
        # physical parameters via positive reparametrization  p = raw^2 + eps
        self.raw = nn.ParameterDict(
            {k: nn.Parameter(torch.tensor(float(v)**0.5)) for k, v in param_init.items()})

    def phys_params(self):
        return {k: self.raw[k]**2 + 1e-6 for k in self.raw}

    def _feat(self, t):
        return t / self.T   # normalize time to [0,1]; the chain rule is handled by autograd

    def forward(self, t, x0):
        return self.net(torch.cat([self._feat(t), x0], dim=1))

def fit_pinn(model, data, rhs_t, steps=4000, lr=1e-3, w=(5.0, 1.0, 1.0), log=500):
    t_d, x0_d, x_d, t_ic, x_ic = data   # all tensors
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for it in range(steps):
        opt.zero_grad()
        loss_data = ((model(t_d, x0_d) - x_d)**2).mean()           # fit the measurements
        tc = t_d.clone().requires_grad_(True)                       # physics residual:
        u = model(tc, x0_d)                                         #   du/dt should equal f(u)
        grads = [torch.autograd.grad(u[:, j].sum(), tc, create_graph=True)[0]
                 for j in range(model.m)]
        du = torch.cat(grads, dim=1)
        loss_phys = ((du - rhs_t(u, model.phys_params()))**2).mean()
        loss_ic = ((model(t_ic, x_ic) - x_ic)**2).mean()           # match initial conditions
        loss = w[0]*loss_data + w[1]*loss_phys + w[2]*loss_ic
        loss.backward(); opt.step()
        if log and it % log == 0:
            print(f'  step {it:5d}  data {loss_data.item():.2e}  phys {loss_phys.item():.2e}')
    return model

In [ ]:
def rhs_t(x, p):
    x1, x2 = x[:,0:1], x[:,1:2]
    prod1 = (hill_act_t(x1,p['L11'],p['U11'],p['th11'],p['d11'])
             + hill_act_t(x2,p['L21'],p['U21'],p['th21'],p['d21']))
    prod2 = (hill_rep_t(x1,p['L12'],p['U12'],p['th12'],p['d12'])
             * hill_act_t(x2,p['L22'],p['U22'],p['th22'],p['d22']))
    return torch.cat([-x1 + prod1, -x2 + prod2], dim=1)

init = {k: (0.05 if k[0]=='L' else 1.0) for k in KEYS}
init.update({k: 5.0 for k in DKEYS})
torch.manual_seed(0)
model = PINN(m=2, param_init=init, T=T, hidden=64, depth=4).to(DEVICE)
data = build_tensors(ts, xs_noisy, x0s)
model = fit_pinn(model, data, rhs_t, steps=6000, lr=1e-3, w=(5.0,1.0,1.0))

pp = {k: float(v.detach()) for k, v in model.phys_params().items()}
print('PINN estimate:', {k: round(pp[k],2) for k in KEYS})
print('PINN lands in node 712:', in_region(pp))

## Morse graph and Conley-index recovery (DSGRN)

There are two criteria per recovered parameter set: exact DSGRN region equality, and label-preserving isomorphism of the **Conley-Morse graph** (recurrent Morse sets, reachability order, Conley-index labels). Region equality implies isomorphic Morse graphs, so the second criterion is weaker than the first.

The cells below compare the recovered and target Morse graphs via `par_index_from_sample` (parameter sample to region index), `DSGRN_utils.ConleyMorseGraph` (region index to Morse graph), and `DSGRN.isomorphic_morse_graphs` (isomorphism of Morse graphs). For the target region we also plot the Morse sets in state space with `DSGRN_utils.PlotMorseSets` and the Conley-Morse graph with `DSGRN_utils.PlotMorseGraph`.

In [ ]:
# Conley-Morse graph for each region, reusing the network and TARGET from Section 1
_mg = {}
def conley_morse(idx):          # region index -> Conley-Morse graph (cached)
    if idx not in _mg:
        _mg[idx] = DSGRN_utils.ConleyMorseGraph(_pg.parameter(idx))[0]
    return _mg[idx]
def morse_recovers(idx, target=TARGET):   # same Conley-Morse graph up to label-preserving iso
    return idx >= 0 and DSGRN.isomorphic_morse_graphs(conley_morse(idx), conley_morse(target))

_cmg = DSGRN_utils.ConleyMorseGraph(_pg.parameter(TARGET))

In [ ]:
DSGRN_utils.PlotMorseSets(*_cmg)

In [ ]:
DSGRN_utils.PlotMorseGraph(_cmg[0])

In [ ]:
# the recovered parameters from the cells above
p_ls = ls_recover(ts, xs_noisy)
print('recovered region index:', region_of(p_ls),
      '| exact region match:', region_of(p_ls) == TARGET,
      '| Morse/Conley match:', morse_recovers(region_of(p_ls)))

# noise sweep: exact-region recovery vs Morse/Conley recovery (least squares)
levels = [0.0, 0.1, 0.25, 0.5]
print(f'{"noise":>6} | {"exact region":>12} | {"Morse/Conley":>12} | {"Morse-ok / miss":>16}')
for ub in levels:
    r = np.random.default_rng(7); reg = mor = gapc = 0
    for _ in range(15):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        idx = region_of(ls_recover(ts, xs_n))
        er = (idx == TARGET); mr = morse_recovers(idx)
        reg += er; mor += mr; gapc += (mr and not er)
    miss = 15 - reg
    print(f'{ub:>6} | {reg:>10}/15 | {mor:>10}/15 | {f"{gapc}/{miss}":>16}')